## Severity Indicators Calculation

* __Fire Incident Rate__:

__Definition__: The number of fire incidents occurring within a specific dissemination area.  
__Purpose__: This indicator helps to understand how frequently fire incidents are happening in different areas. High fire incident rates could indicate a need for better fire prevention measures and public education.  
__Calculation__: The total number of fire incidents in a given area.

* __Percentage of Non-Working Smoke Alarms in Fires__:

__Definition__: The proportion of fire incidents where smoke alarms were not working out of the total fire incidents.  
__Purpose__: This indicator highlights the effectiveness of smoke alarm maintenance and installation in different areas. A high percentage of non-working smoke alarms could suggest a need for better public awareness campaigns or stricter regulations on smoke alarm installations.  
__Calculation__: (Number of fire incidents with non-working smoke alarms / Total number of fire incidents) * 100.

* __Civilian Injuries Rate per Incident__:

__Definition__: The average number of civilian injuries per fire incident.  
__Purpose__: This indicator measures the impact of fire incidents on human health and safety. High rates of civilian injuries per incident can indicate the severity of fires and the effectiveness of emergency response services.  
__Calculation__: Total number of civilian injuries / Total number of fire incidents.  

* __Casualty Rate per Incident__:

__Definition__: The average number of civilian fatalities per fire incident.  
__Purpose__: This indicator provides insight into the deadliness of fire incidents in different areas. Higher casualty rates may indicate more severe fire incidents or delayed emergency response times.  
__Calculation__: Total number of civilian fatalities / Total number of fire incidents.  

* __Property Damage Cost per Fire Incident__:

__Definition__: The average monetary value of property damage per fire incident.  
__Purpose__: This indicator assesses the financial impact of fire incidents. High property damage costs per incident can indicate more severe fires, possibly due to the type of buildings affected or the effectiveness of fire control measures.  
__Calculation__: Total property damage cost / Total number of fire incidents.  

* __Average Response Time__:

__Definition__: The average time taken for emergency services to respond to fire incidents.  
__Purpose__: This indicator evaluates the efficiency of emergency response services. Shorter response times generally lead to less severe outcomes in fire incidents.  
__Calculation__: Sum of response times / Total number of fire incidents.  

In [8]:
import pandas as pd
from scipy.stats import zscore

In [3]:
df = pd.read_excel("../Cleaned_Dataset/new_DA.xlsx")
df.head()

,unique_Id,inci_no,inci_type,incident_category,Incident_Type_Description,Property_Loss_Value,Content_Loss_Value,Property_Value,Content_Value,Civilian_Fatal,...,Property_Damage_Category,Property_Damage_Description,Received_Datetime,Dispatched_Datetime,Arrival_Datetime,Cleared_Datetime,DAUID,Longitude,Latitude,DAUID_new
0,397231,20-0006971,89,I,Other Medical/Resuscitator Call ...,0,0,0,0,0,...,350,"Hotel, Motel, Lodging - 4 or more guests or su...",2020-10-31 05:21:00,2020-10-31 05:21:00,2020-10-31 05:27:00,2020-10-31 05:33:00,35431013.0,-79.697299,44.364071,35431014
1,397468,20-0007034,89,I,Other Medical/Resuscitator Call ...,0,0,0,0,0,...,320,Multi-Unit Dwelling - Over 12 Units ...,2020-11-02 17:48:00,2020-11-02 17:48:00,2020-11-02 17:53:00,2020-11-02 18:22:00,35431321.0,-79.699290,44.354853,35430680
2,397549,20-0007052,89,I,Other Medical/Resuscitator Call ...,0,0,0,0,0,...,350,"Hotel, Motel, Lodging - 4 or more guests or su...",2020-11-03 15:16:00,2020-11-03 15:16:00,2020-11-03 15:22:00,2020-11-03 15:39:00,35431013.0,-79.697299,44.364071,35431014
3,398004,20-0007183,89,I,Other Medical/Resuscitator Call ...,0,0,0,0,0,...,320,Multi-Unit Dwelling - Over 12 Units ...,2020-11-09 02:53:00,2020-11-09 02:53:00,2020-11-09 02:59:00,2020-11-09 03:06:00,35431321.0,-79.699290,44.354853,35430680
4,398196,20-0007238,89,I,Other Medical/Resuscitator Call ...,0,0,0,0,0,...,350,"Hotel, Motel, Lodging - 4 or more guests or su...",2020-11-11 00:38:00,2020-11-11 00:38:00,2020-11-11 00:45:00,2020-11-11 00:51:00,35431013.0,-79.697299,44.364071,35431014


In [28]:
datetime_format = '%d/%m/%Y %I:%M:%S %p'

# Convert columns to datetime
df['Received_Datetime'] = pd.to_datetime(df['Received_Datetime'], format=datetime_format)
df['Dispatched_Datetime'] = pd.to_datetime(df['Dispatched_Datetime'], format=datetime_format)
df['Arrival_Datetime'] = pd.to_datetime(df['Arrival_Datetime'], format=datetime_format)
df['Cleared_Datetime'] = pd.to_datetime(df['Cleared_Datetime'], format=datetime_format)

# Calculate response time (in minutes)
df['Response_Time'] = (df['Arrival_Datetime'] - df['Received_Datetime']).dt.total_seconds() / 60

# Calculate total incidents per DAUID
total_incidents = df.groupby('DAUID_new').size().reset_index(name='Fire_Incidents')

# Group by 'DAUID_new' and calculate sums and means
grouped = df.groupby('DAUID_new').agg({
    'Civilian_Fatal': 'sum',
    'Civilian_Injuries': 'sum',
    'Property_Loss_Value': 'sum',
    'Response_Time': 'mean',
    # 'Smoke_Alarm_Status': lambda x: (x == 'Not Working').sum() / len(x) * 100  # percentage of non-working smoke alarms
}).reset_index()

# Merge the two DataFrames on 'DAUID_new'
grouped = pd.merge(grouped, total_incidents, left_on='DAUID_new', right_on='DAUID_new')

# Calculate the casualty rate per incident
grouped['Casualty_Rate_Per_Incident'] = (grouped['Civilian_Fatal'] / grouped['Fire_Incidents']).round(2)

# Calculate the civilian injuries rate per incident
grouped['Injuries_Rate_Per_Incident'] = (grouped['Civilian_Injuries'] / grouped['Fire_Incidents']).round(2)

# Calculate the property damage cost per fire incident
grouped['Property_Damage_Cost_Per_Incident'] = (grouped['Property_Loss_Value'] / grouped['Fire_Incidents']).round(2)

# Calculate the average response time
grouped['Average_Response_Time'] = grouped['Response_Time'].round(2)

# Rename the columns for clarity
grouped.rename(columns={
    'DAUID_new': 'DAUID'
    # 'Smoke_Alarm_Status': 'Percentage_Non_Working_Smoke_Alarms'
}, inplace=True)

# Load the additional risk indicator from another CSV file
population_df = pd.read_csv('../cleaned_dataset/population_aged_above_65.csv')

# Merge the population data with the grouped data
grouped = pd.merge(grouped, population_df, on='DAUID', how='left')

# Rename the columns for clarity
grouped.rename(columns={'percentage_above_aged_65': 'Population_Aged_65_Plus'}, inplace=True)

# Standardize the indicators using z-score
indicators = [
    'Fire_Incidents',
    'Injuries_Rate_Per_Incident',
    'Casualty_Rate_Per_Incident',
    'Property_Damage_Cost_Per_Incident',
    'Average_Response_Time',
    'Population_Aged_65_Plus'
]

for indicator in indicators:
    grouped[f'{indicator}_Z'] = zscore(grouped[indicator])
    
print(grouped.head())

# Identify top 10% DAs for each indicator
for indicator in indicators:
    threshold = grouped[f'{indicator}_Z'].quantile(0.9)
    grouped[f'{indicator}_High_Risk'] = (grouped[f'{indicator}_Z'] >= threshold).astype(int)

# Create composite severity score
grouped['Composite_Severity_Score'] = grouped[[f'{indicator}_High_Risk' for indicator in indicators]].sum(axis=1)

# Categorize DAs based on composite severity score
def categorize_severity(score):
    if score >= 5:
        return '5: Extreme Severity'
    elif score == 4:
        return '4: Very High Severity'
    elif score == 3:
        return '3: High Severity'
    elif score == 2:
        return '2: Moderate Severity'
    else:
        return '1: Low Severity'

grouped['Severity_Category'] = grouped['Composite_Severity_Score'].apply(categorize_severity)

# Display the result
print(grouped)

# Save the result to a CSV file
grouped.to_csv('../cleaned_dataset/severity_indicators_by_dguid.csv', index=False, columns=[
    'DAUID', 
    'Fire_Incidents',
    'Injuries_Rate_Per_Incident', 
    'Casualty_Rate_Per_Incident', 
    'Average_Response_Time',
    'Property_Damage_Cost_Per_Incident',
    'Population_Aged_65_Plus',
    'Composite_Severity_Score',
    'Severity_Category'
])

print("Data saved to severity_indicators_by_dguid.csv")

      DAUID  Civilian_Fatal  Civilian_Injuries  Property_Loss_Value  \
0  35430639               0                  0                    0   
1  35430647               0                  0                15000   
2  35430648               0                  0                 6000   
3  35430649               0                  0                81000   
4  35430650               0                  0                    0   

   Response_Time  Fire_Incidents  Casualty_Rate_Per_Incident  \
0       9.000000               7                         0.0   
1       7.632812             130                         0.0   
2       7.662791              87                         0.0   
3       7.115942              70                         0.0   
4       7.400000              10                         0.0   

   Injuries_Rate_Per_Incident  Property_Damage_Cost_Per_Incident  \
0                         0.0                               0.00   
1                         0.0                       

In [31]:
total_incidents.rename(columns={'DAUID_new': 'DAUID'}, inplace=True)
# Find common DAUIDs
common_dauid = pd.merge(total_incidents, population_df, on='DAUID')
common_count = len(common_dauid)

# Find DAUIDs only in CAD dataframe
only_cad_dauid = total_incidents[~total_incidents['DAUID'].isin(population_df['DAUID'])]
only_cad_count = len(only_cad_dauid)

# Find DAUIDs only in Population dataframe
only_population_dauid = population_df[~population_df['DAUID'].isin(total_incidents['DAUID'])]
only_population_count = len(only_population_dauid)

# Display the counts
print(f"Common DAUID count: {common_count}")
print(f"DAUIDs only in CAD dataframe count: {only_cad_count}")
print(f"DAUIDs only in Population dataframe count: {only_population_count}")

# Display the DAUIDs for reference
print("Common DAUIDs:")
print(common_dauid['DAUID'])
print("DAUIDs only in CAD dataframe:")
print(only_cad_dauid['DAUID'])
print("DAUIDs only in Population dataframe:")
print(only_population_dauid['DAUID'])

Common DAUID count: 241
DAUIDs only in CAD dataframe count: 5
DAUIDs only in Population dataframe count: 2
Common DAUIDs:
0      35430647
1      35430648
2      35430649
3      35430650
4      35430651
         ...   
236    35431370
237    35431371
238    35431372
239    35431373
240    35431380
Name: DAUID, Length: 241, dtype: int64
DAUIDs only in CAD dataframe:
0      35430639
64     35430715
65     35430961
197    35431133
244    35431377
Name: DAUID, dtype: int64
DAUIDs only in Population dataframe:
35    35430682
60    35430710
Name: DAUID, dtype: int64
